In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from dataclasses import dataclass, replace
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Visual style & palette constants
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.linewidth']   = 0.8
plt.rcParams['grid.color']       = '#cccccc'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5

C_IPR = '#1f77b4'   # deep blue
C_VLP = '#d62728'   # crimson
C_OP  = '#2ca02c'   # forest green
C_GAS = '#ff7f0e'   # amber
C_AOF = '#9467bd'   # purple
C_FB  = '#8c564b'   # brown
C_WC  = '#17becf'   # cyan/teal


In [ ]:
# WELL TYPE SELECTOR  ---  run this cell to configure your well.

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, PillowWriter
import os, io, tempfile, base64

# Global configuration
WELL_CONFIG = dict(well_type='Vertical', angle_deg=90.0, survey=None)

# ── Standard API 5CT Tubing Catalogues (by well type) ─────────────
# Each entry: { 'label': nominal_label, 'OD_in': OD, 'ID_in': ID,
#               'weight_lbft': weight, 'default': True/False }
# The entry marked default=True is used when the user doesn't specify.
#
# Reference: API Specification 5CT; industry best-practice selection
# guidelines for vertical, horizontal, directional, and ERD wells.

TUBING_CATALOG = {
    # ----------- VERTICAL wells -----------
    # Conventional choice: 2-7/8" to 3-1/2" for moderate-rate oil wells.
    # Default: 2-7/8" (6.5 lb/ft, EUE) — workhorse of onshore verticals,
    # good balance of friction and liquid-loading prevention.
    'Vertical': [
        {'label': '1.900" (1.610 in ID)', 'OD_in': 1.900, 'ID_in': 1.610, 'weight_lbft': 2.75, 'default': False},
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': True},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': False},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
    ],
    # ----------- HORIZONTAL wells -----------
    # Horizontals favour larger tubing to minimise frictional pressure
    # drop along the long lateral. 3-1/2" or 4-1/2" are common.
    # Default: 3-1/2" (9.3 lb/ft) — most-used in unconventional laterals.
    'Horizontal': [
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': False},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': True},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
        {'label': '4-1/2" (4.000 in ID)', 'OD_in': 4.500, 'ID_in': 4.000, 'weight_lbft': 11.60, 'default': False},
    ],
    # ----------- DIRECTIONAL wells -----------
    # Directional (deviated / S-shape / J-shape) wells see higher
    # drag & torque.  2-7/8" to 3-1/2" most common; default 2-7/8".
    'Directional': [
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': True},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': False},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
    ],
    # ----------- CUSTOM trajectory -----------
    # Same catalogue as Directional; user presumably knows their well.
    'Custom': [
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': True},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': False},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
    ],
}

def get_default_tubing(well_type='Vertical'):
    """Return the default tubing entry for a well type."""
    cat = TUBING_CATALOG.get(well_type, TUBING_CATALOG['Vertical'])
    for t in cat:
        if t.get('default'):
            return t
    return cat[0]

def get_tubing_sensitivity_list(well_type='Vertical'):
    """Return {label: ID_in} dict for all tubing sizes for a well type."""
    cat = TUBING_CATALOG.get(well_type, TUBING_CATALOG['Vertical'])
    return {t['label']: t['ID_in'] for t in cat}


def _min_curve(md, inc, azi):
    md = np.asarray(md,float); inc = np.asarray(inc,float); azi = np.asarray(azi,float)
    n = len(md); tvd=np.zeros(n); north=np.zeros(n); east=np.zeros(n)
    for i in range(1, n):
        dmd=md[i]-md[i-1]; a1,a2=np.radians(inc[i-1]),np.radians(inc[i])
        f1,f2=np.radians(azi[i-1]),np.radians(azi[i])
        dl=np.arccos(np.clip(np.cos(a2-a1)-np.sin(a1)*np.sin(a2)*(1-np.cos(f2-f1)),-1,1))
        rf=(2/dl*np.tan(dl/2)) if dl>1e-9 else 1.0
        tvd[i]=tvd[i-1]+dmd/2*(np.cos(a1)+np.cos(a2))*rf
        north[i]=north[i-1]+dmd/2*(np.sin(a1)*np.cos(f1)+np.sin(a2)*np.cos(f2))*rf
        east[i]=east[i-1]+dmd/2*(np.sin(a1)*np.sin(f1)+np.sin(a2)*np.sin(f2))*rf
    return tvd, north, east

def build_survey_df(md, inc, azi, unit='ft'):
    md=np.asarray(md,float); inc=np.asarray(inc,float); azi=np.asarray(azi,float)
    if unit.lower() in ('m','meter','metre','meters','metres'): md=md*3.28084
    tvd, north, east = _min_curve(md, inc, azi)
    return pd.DataFrame({'MD_ft':md,'TVD_ft':tvd,'INC_survey_deg':inc,
                         'INC_code_deg':90.0-inc,'AZI_deg':azi,'North_ft':north,'East_ft':east})

def plot_survey_with_3d_gif(sv, out_widget=None):
    import contextlib
    ctx = out_widget if out_widget is not None else contextlib.nullcontext()
    with ctx:
        clear_output(wait=True)
        # 2-D views
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.plot(sv.East_ft, sv.North_ft, 'b.-', lw=1.5, ms=3)
        ax1.set(xlabel='East (ft)', ylabel='North (ft)', title='Plan View (Top-down)')
        ax1.set_aspect('equal', 'box'); ax1.grid(True, alpha=0.3)
        dep = np.hypot(sv.East_ft, sv.North_ft)
        ax2.plot(dep, -sv.TVD_ft, 'b.-', lw=1.5, ms=3)
        ax2.set(xlabel='Horizontal Departure (ft)', ylabel='TVD (ft)', title='Vertical Section')
        ax2.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
        # 3-D rotating GIF  -- 72 frames x 5deg = 360deg, 6 fps => ~12s slow loop
        try:
            fig3d = plt.figure(figsize=(5.5, 4.5), dpi=80)
            ax3d = fig3d.add_subplot(111, projection='3d')
            ax3d.plot(sv.East_ft, sv.North_ft, -sv.TVD_ft, color='#1f77b4', lw=2.5, label='Trajectory')
            ax3d.scatter([sv.East_ft.iloc[0]],[sv.North_ft.iloc[0]],[-sv.TVD_ft.iloc[0]],
                         color='green', s=60, zorder=5, label='Wellhead')
            ax3d.scatter([sv.East_ft.iloc[-1]],[sv.North_ft.iloc[-1]],[-sv.TVD_ft.iloc[-1]],
                         color='red', s=60, zorder=5, label='TD')
            ax3d.set_xlabel('East (ft)', fontsize=8.5)
            ax3d.set_ylabel('North (ft)', fontsize=8.5)
            ax3d.set_zlabel('Depth (ft)', fontsize=8.5)
            ax3d.set_title('3D Well Trajectory (Rotating View)', fontsize=10.5, fontweight='bold')
            ax3d.legend(loc='upper right', fontsize=8)
            frames = range(0, 360, 5)
            def _upd(f): ax3d.view_init(elev=20, azim=f); return ax3d,
            anim = FuncAnimation(fig3d, _upd, frames=frames, interval=167)
            with tempfile.NamedTemporaryFile(suffix='.gif', delete=False) as tmp:
                anim.save(tmp.name, writer=PillowWriter(fps=6))
                tmp_path = tmp.name
            plt.close(fig3d)
            with open(tmp_path, 'rb') as f:
                gif_b64 = base64.b64encode(f.read()).decode()
            os.remove(tmp_path)
            display(HTML(
                '<div style="margin-top:10px">'
                '<b style="font-size:13px;color:#333">3D Trajectory Animation:</b><br/>'
                '<img src="data:image/gif;base64,' + gif_b64 + '" '
                'style="border:1px solid #ddd;border-radius:6px;margin-top:5px;'
                'max-width:480px;box-shadow:0 2px 6px rgba(0,0,0,.1)"/></div>'
            ))
        except Exception as e:
            print(f'Note: 3D GIF could not be generated: {e}')

_S    = {'description_width': '130px'}
_S_short = {'description_width': '50px'}
_S_col   = {'description_width': '70px'}
_W160 = widgets.Layout(width='165px')
_W240 = widgets.Layout(width='245px')
_W300 = widgets.Layout(width='305px')

wt_dd      = widgets.Dropdown(options=['Vertical','Horizontal','Directional','Custom'],
                 value='Vertical', description='Well Type:', style=_S, layout=_W300)
status_lbl = widgets.HTML('<span style="color:green;font-weight:bold">✔  Vertical  —  angle = 90°</span>')
out_main   = widgets.Output()

# DIRECTIONAL
dir_file  = widgets.FileUpload(accept='.xlsx,.xls,.csv', multiple=False,
               description='📂 Select File', style=_S, layout=_W300)
dir_sheet = widgets.Dropdown(description='Sheet / Table:', options=[], style=_S, layout=_W300, disabled=True)
dir_unit  = widgets.Dropdown(options=['ft','m'], value='ft', description='Depth unit:', style=_S, layout=_W300)
dir_md_c  = widgets.Dropdown(description='MD col:',  options=[], style=_S_col, layout=widgets.Layout(width='220px'), disabled=True)
dir_inc_c = widgets.Dropdown(description='INC col:', options=[], style=_S_col, layout=widgets.Layout(width='220px'), disabled=True)
dir_azi_c = widgets.Dropdown(description='AZI col:', options=[], style=_S_col, layout=widgets.Layout(width='220px'), disabled=True)
dir_load  = widgets.Button(description='⬆  Load & Compute TVD',
               button_style='primary', layout=widgets.Layout(width='220px'), disabled=True)
dir_out   = widgets.Output()
_cached_excel = None
_cached_csv   = None

def _guess_col(opts, patterns):
    for o in opts:
        ol = str(o).strip().lower()
        for p in patterns:
            if ol == p or p in ol: return o
    return opts[0] if opts else None

def _update_cols(cols):
    cs = [str(c) for c in cols]
    if not cs: return
    dir_md_c.options  = cs; dir_md_c.disabled  = False
    dir_inc_c.options = cs; dir_inc_c.disabled = False
    g = _guess_col(cs, ['md','measured','depth']);     dir_md_c.value  = g if g else cs[0]
    g = _guess_col(cs, ['inc','incl','inclination']);  dir_inc_c.value = g if g else cs[0]
    ao = ['(None / 0°)'] + cs
    dir_azi_c.options = ao; dir_azi_c.disabled = False
    g = _guess_col(cs, ['azi','azim','azimuth','dir']); dir_azi_c.value = g if g else '(None / 0°)'

def _on_sheet(change):
    if _cached_excel and change['new']:
        try:   _update_cols(list(_cached_excel.parse(change['new'], nrows=3).columns))
        except Exception as e:
            with dir_out: print(f'⚠  {e}')

dir_sheet.observe(_on_sheet, names='value')

def _on_upload(change):
    global _cached_excel, _cached_csv
    with dir_out:
        clear_output()
        val = dir_file.value
        if not val: return
        try:
            if isinstance(val, dict):
                fname = list(val.keys())[0]; content = val[fname]['content']
            else:
                fname = val[0]['name'];       content = val[0]['content']
            fb  = content if isinstance(content, bytes) else content.tobytes()
            ext = os.path.splitext(fname)[1].lower()
            if ext in ('.xlsx','.xls','.xlsm','.xlsb'):
                _cached_excel = pd.ExcelFile(io.BytesIO(fb)); _cached_csv = None
                sn = _cached_excel.sheet_names
                dir_sheet.options = sn; dir_sheet.disabled = False
                if sn:
                    dir_sheet.value = sn[0]
                    _update_cols(list(_cached_excel.parse(sn[0], nrows=3).columns))
                print(f'✔  {repr(fname)} loaded — {len(sn)} sheet(s). Select sheet & columns, then click Load.')
            elif ext == '.csv':
                _cached_excel = None; _cached_csv = pd.read_csv(io.BytesIO(fb))
                dir_sheet.options = ['(CSV — no sheets)']; dir_sheet.disabled = True
                _update_cols(list(_cached_csv.columns))
                print(f'✔  CSV {repr(fname)} loaded. Check columns and click Load.')
            else:
                print(f'⚠  Unsupported format: {ext}'); return
            dir_load.disabled = False
        except Exception as e:
            import traceback; print(f'✘  {e}'); traceback.print_exc()

dir_file.observe(_on_upload, names='value')

def _load_dir(btn):
    with dir_out:
        clear_output()
        if _cached_excel is None and _cached_csv is None:
            print('⚠  Upload a file first.'); return
        try:
            df = _cached_excel.parse(dir_sheet.value) if _cached_excel else _cached_csv
            mc, ic, ac = str(dir_md_c.value), str(dir_inc_c.value), str(dir_azi_c.value)
            miss = [c for c in [mc, ic] if c not in df.columns]
            if miss:
                print(f'⚠  Columns not found: {miss}\n   Available: {list(df.columns)}'); return
            md_a  = df[mc].values.astype(float)
            inc_a = df[ic].values.astype(float)
            azi_a = df[ac].values.astype(float) if (ac in df.columns and ac!='(None / 0°)') else np.zeros(len(md_a))
            sv = build_survey_df(md_a, inc_a, azi_a, unit=dir_unit.value)
            WELL_CONFIG.update(well_type='Directional', angle_deg=None, survey=sv)
            print(f'✔  Survey: {len(sv)} stations  |  MD 0→{sv.MD_ft.iloc[-1]:,.0f} ft  |  TVD 0→{sv.TVD_ft.iloc[-1]:,.0f} ft')
            print(f'   Max inclination: {sv.INC_survey_deg.max():.1f}°   (rendering plots, please wait…)')
            plot_survey_with_3d_gif(sv, out_widget=dir_out)
            status_lbl.value = (
                f'<span style="color:green;font-weight:bold">✔  Directional — '
                f'{len(sv)} stations | Max inc {sv.INC_survey_deg.max():.1f}° | '
                f'TVD {sv.TVD_ft.iloc[-1]:,.0f} ft</span>')
        except Exception as e:
            import traceback; print(f'✘  {e}'); traceback.print_exc()

dir_load.on_click(_load_dir)

dir_panel = widgets.VBox([
    widgets.HTML('<b style="font-size:14px">📂 Directional Survey — File Input</b>'),
    widgets.HTML('<span style="color:#555;font-size:12px">INC: 0° = vertical, 90° = horizontal</span>'),
    dir_file,
    widgets.HBox([dir_sheet, dir_unit]),
    widgets.HBox([dir_md_c, dir_inc_c, dir_azi_c]),
    dir_load, dir_out,
], layout=widgets.Layout(border='1px solid #ccc', padding='12px', margin='8px 0'))

# CUSTOM
_cst_rows = []

cst_unit = widgets.Dropdown(options=['ft','m'], value='ft', description='Depth unit:', style=_S, layout=_W300)
cst_md   = widgets.FloatText(value=0, description='MD:',    style=_S_short, layout=widgets.Layout(width='190px'))
cst_inc  = widgets.FloatText(value=0, description='INC°:', style=_S_short, layout=widgets.Layout(width='190px'))
cst_azi  = widgets.FloatText(value=0, description='AZI°:', style=_S_short, layout=widgets.Layout(width='190px'))
cst_add  = widgets.Button(description='➕ Add Station',      button_style='success', layout=_W160)
cst_del  = widgets.Button(description='🗑 Del Last',      button_style='warning', layout=_W160)
cst_clr  = widgets.Button(description='✖ Clear All',         button_style='danger',  layout=_W160)
cst_apl  = widgets.Button(description='✔  Apply & Preview',  button_style='primary',
               layout=widgets.Layout(width='210px'))
cst_tbl  = widgets.HTML('<i style="color:#888">No stations yet. Add ≥2 then click Apply & Preview.</i>')
cst_out  = widgets.Output()

def _cst_refresh_table():
    if not _cst_rows:
        cst_tbl.value = '<i style="color:#888">No stations yet. Add ≥2 then click Apply & Preview.</i>'; return
    h = ('<table border="1" style="border-collapse:collapse;font-size:12px;font-family:monospace;margin:4px 0">'
         '<tr style="background:#f0f0f0"><th style="padding:3px 10px">#</th>'
         '<th style="padding:3px 10px">MD</th><th style="padding:3px 10px">INC (°)</th>'
         '<th style="padding:3px 10px">AZI (°)</th></tr>')
    for j, r in enumerate(_cst_rows, 1):
        h += (f'<tr><td style="padding:3px 10px">{j}</td>'
              f'<td style="padding:3px 10px">{r["md"]:.1f}</td>'
              f'<td style="padding:3px 10px">{r["inc"]:.2f}</td>'
              f'<td style="padding:3px 10px">{r["azi"]:.2f}</td></tr>')
    cst_tbl.value = h + '</table>'

def _cst_add(btn):
    _cst_rows.append({'md': cst_md.value, 'inc': cst_inc.value, 'azi': cst_azi.value})
    _cst_rows.sort(key=lambda x: x['md'])
    _cst_refresh_table()   # only update table, no plot

def _cst_del(btn):
    if _cst_rows: _cst_rows.pop()
    _cst_refresh_table()

def _cst_clr(btn):
    _cst_rows.clear(); _cst_refresh_table()
    with cst_out: clear_output()

def _cst_apply(btn):
    with cst_out:
        if len(_cst_rows) < 2:
            clear_output(); print('⚠  Add at least 2 stations first.'); return
        md_a = np.array([r['md']  for r in _cst_rows])
        ic_a = np.array([r['inc'] for r in _cst_rows])
        az_a = np.array([r['azi'] for r in _cst_rows])
        sv = build_survey_df(md_a, ic_a, az_a, unit=cst_unit.value)
        WELL_CONFIG.update(well_type='Custom', angle_deg=None, survey=sv)
        status_lbl.value = (
            f'<span style="color:green;font-weight:bold">✔  Custom — '
            f'{len(sv)} stations | Max inc {sv.INC_survey_deg.max():.1f}° | '
            f'TVD {sv.TVD_ft.iloc[-1]:,.0f} ft</span>')
        print('✔  Trajectory applied — rendering plots, please wait…')
        plot_survey_with_3d_gif(sv, out_widget=cst_out)

cst_add.on_click(_cst_add); cst_del.on_click(_cst_del)
cst_clr.on_click(_cst_clr); cst_apl.on_click(_cst_apply)

cust_panel = widgets.VBox([
    widgets.HTML('<b style="font-size:14px">✏  Custom Well — Interactive Trajectory Builder</b>'),
    widgets.HTML('<span style="color:#555;font-size:12px">INC: 0°=vertical, 90°=horizontal | '
                 'Stations sorted by MD. Click <b>Apply & Preview</b> to compute + visualise.</span>'),
    cst_unit,
    widgets.HBox([cst_md, cst_inc, cst_azi]),
    widgets.HBox([cst_add, cst_del, cst_clr]),
    cst_tbl,
    cst_apl,
    cst_out,
], layout=widgets.Layout(border='1px solid #ccc', padding='12px', margin='8px 0'))

def _on_wt_change(change):
    wt = change['new']
    with out_main:
        clear_output(wait=True)
        if wt == 'Vertical':
            WELL_CONFIG.update(well_type='Vertical', angle_deg=90.0, survey=None)
            status_lbl.value = '<span style="color:green;font-weight:bold">✔  Vertical  —  angle = 90°</span>'
        elif wt == 'Horizontal':
            WELL_CONFIG.update(well_type='Horizontal', angle_deg=0.0, survey=None)
            status_lbl.value = '<span style="color:green;font-weight:bold">✔  Horizontal  —  angle = 0°</span>'
        elif wt == 'Directional':
            WELL_CONFIG.update(well_type='Directional', angle_deg=None, survey=None)
            status_lbl.value = '<span style="color:darkorange;font-weight:bold">⏳  Directional — load survey file below</span>'
            display(dir_panel)
        elif wt == 'Custom':
            WELL_CONFIG.update(well_type='Custom', angle_deg=None, survey=None)
            status_lbl.value = '<span style="color:darkorange;font-weight:bold">⏳  Custom — build trajectory below</span>'
            display(cust_panel)

wt_dd.observe(_on_wt_change, names='value')

display(widgets.VBox([
    widgets.HTML(
        '<h3 style="margin:6px 0;color:#333">⚙  Well Type Configuration</h3>'
        '<p style="color:#555;font-size:12px;margin:0 0 6px 0">'
        'Select a well type, configure the trajectory, then re-run the VLP cells below.</p>'),
    wt_dd, status_lbl, out_main,
]))

# sanity check
print(f"[dbg] config → type={WELL_CONFIG['well_type']!r},"
      f" Pr={WELL_CONFIG['Pr']} psia, GOR={WELL_CONFIG['GOR']} scf/STB")


In [ ]:
# PVT Correlations — Oil & Gas Fundamentals (Standing, 1947; Vasquez & Beggs, 1980)

def oil_sg(API):
    """Stock-tank oil specific gravity from API gravity."""
    return 141.5 / (131.5 + API)

def standing_rs(P, gg, API, T):
    """Standing (1947) solution GOR, scf/STB. Valid for P <= Pb."""
    a = 0.0125 * API - 0.00091 * T
    return gg * ((P / 18.2 + 1.4) * 10 ** a) ** 1.2048

def standing_pb(Rsb, gg, API, T):
    """Standing (1947) bubble-point pressure, psia (inverse of standing_rs)."""
    a = 0.0125 * API - 0.00091 * T
    return 18.2 * ((Rsb / gg) ** (1 / 1.2048) * 10 ** (-a) - 1.4)

def standing_bo_sat(Rs, gg, go, T):
    """Standing (1947) saturated oil FVF, rb/STB."""
    F = Rs * (gg / go) ** 0.5 + 1.25 * T
    return 0.9759 + 0.00012 * F ** 1.2

def vasquez_beggs_co(P, Rsb, gg, API, T):
    """Vasquez & Beggs (1980) undersaturated oil isothermal compressibility, 1/psi."""
    return (-1433 + 5 * Rsb + 17.2 * T - 1180 * gg + 12.61 * API) / (1e5 * P)

def oil_pvt(P, gg, API, T, Pb, Rsb):
    """Return (Rs, Bo, rho_o[lbm/ft3]) at pressure P, honoring saturated/undersaturated regime."""
    go = oil_sg(API)
    if P < Pb:
        Rs = standing_rs(P, gg, API, T)
        Bo = standing_bo_sat(Rs, gg, go, T)
    else:
        Rs = Rsb
        Bob = standing_bo_sat(Rsb, gg, go, T)
        co = vasquez_beggs_co(P, Rsb, gg, API, T)
        Bo = Bob * np.exp(co * (Pb - P))
    rho_o = (350.17 * go + 0.0764 * Rs * gg) / (5.615 * Bo)   # lbm/ft3
    return Rs, Bo, rho_o

def gas_pseudocritical(gg):
    """Sutton (1985) pseudo-critical temperature (degR) and pressure (psia)."""
    Tpc = 169.2 + 349.5 * gg - 74.0 * gg ** 2
    Ppc = 756.8 - 131.0 * gg - 3.6 * gg ** 2
    return Tpc, Ppc

def _z_dak(Ppr, Tpr):
    """Dranchuk & Abou-Kassem (1975) real-gas Z-factor, solved implicitly."""
    A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11 = (0.3265,-1.0700,-0.5339,0.01569,-0.05165,
        0.5475,-0.7361,0.1844,0.1056,0.6134,0.7210)
    def resid(Z):
        rho_r = 0.27 * Ppr / (Z * Tpr)
        c1 = A1 + A2/Tpr + A3/Tpr**3 + A4/Tpr**4 + A5/Tpr**5
        c2 = A6 + A7/Tpr + A8/Tpr**2
        c3 = A9 * (A7/Tpr + A8/Tpr**2)
        rhs = (1 + c1*rho_r + c2*rho_r**2 - c3*rho_r**5 +
               A10*(1+A11*rho_r**2)*(rho_r**2/Tpr**3)*np.exp(-A11*rho_r**2))
        return Z - rhs
    try:
        return brentq(resid, 0.2, 3.0, xtol=1e-8, maxiter=200)
    except ValueError:
        return 0.9

def gas_z(P, T, gg):
    """Real-gas compressibility factor Z(P, T, gamma_g)."""
    Tpc, Ppc = gas_pseudocritical(gg)
    return _z_dak(P / Ppc, (T + 460) / Tpc)

def gas_bg(P, T, Z):
    """Gas formation volume factor Bg, rcf/scf."""
    return 0.02827 * Z * (T + 460) / P

def gas_rho(P, T, Z, gg):
    """Gas density, lbm/ft3."""
    return 2.7 * gg * P / (Z * (T + 460))
